In [81]:
import numpy as np


Here is some code to do forward propagation of a DNN.

In [83]:
def relu(z):
    # relu activation
    if z<=0:
        return 0
    else:
        return z

def drelu(z):
    # relu derivative
    if z<=0:
        return 0
    else: 
        return 1

def sigmoid(z):
    # sigmoid function
    return 1/(1+np.exp(z))

def dsigmoid(z):
    # derivative of the sigmoid function
    return sigmoid(z)*(1-sigmoid(z))

def error_f(y, t):
    # compute the error function
    return np.sum(y*np.log(t) + (1-y)*np.log(1-t)  )

def gd_step(x, y, w1, b1, w2, b2, eta):
    # x = inputs
    # y = output (single binary vector)
    # w = weights
    # b = biases
    # eta = learning rate > 0
    
    # forward prop
    a1 = w1 @ x + b1 # preactivation
    z1 = np.maximum(a1,0) # relu activation
    a2 = np.dot(w2,z1) + b2 # out preactivation
    t = sigmoid(a2)

    # backpropagation part 1: compute deltas
    delta2 = dsigmoid(a2)*( y/t - (1-y)/(1-t) ) # deltas for the outside level
    delta1 = (delta2 * w2) * np.maximum(a1,0)/a1 # deltas for the hiden units

    # compute derivatives of weights
    dw2 = delta2 * z1 # gradient for w2
    db2 = delta2 # gradient for b2
    dw1 = np.outer(delta1, x) # gradient for w1
    db1 = delta1 # gradient for b1
    
    # update values
    w2 -= eta * dw2
    b2 -= eta * db2
    w1 -= eta * dw1
    b1 -= eta * db1
    
    # forward prop (for error function)
    a1 = w1 @ x + b1 # preactivation
    z1 = np.maximum(a1,0) # relu activation
    a2 = np.dot(w2,z1) + b2 # out preactivation
    t = sigmoid(a2)
    # current value of error function
    err_f = error_f(y,t)
    
    # return
    return w1, b1, w2, b2, err_f

Let's evaluate these functions on simulated data.

In [118]:
# preliminaries
p = 10 # dimension of output
m = 5 # dimension of hidden units
n = 1000 # sample size
# simulating data
np.random.seed(1996)
x = np.random.normal(0,1,(n,p))
w1 = np.random.normal(0,1,(m,p)) # weights
b1 = np.random.normal(0,1, m)
w2 = np.random.normal(0,1,m)
b2 = np.random.normal(0,1)
a1 = x @ w1.T + b1 # preactivation
z1 = np.maximum(a1,0) # relu activation
a2 = z1 @ w2 + b2 # out preactivation
t = sigmoid(a2)
y = np.where(t>0.5,1,0)

#gradient descent
#initialize
np.random.seed(2000)
w1_gd = np.random.normal(0,1,(m,p)) # weights
b1_gd = np.random.normal(0,1, m)
w2_gd = np.random.normal(0,1,m)
b2_gd = np.random.normal(0,1)
# gd parameters
eta = 1e-3 # learning rate
R = 50000 # number of iterations
err = np.array([])
i = 0 # index
for r in range(R):
    w1_gd, b1_gd, w2_gd, b2_gd, err_f = gd_step(x[i], y[i], w1_gd, b1_gd, w2_gd, b2_gd, eta)
    err = np.append(err,err_f)
    if i ==n-1:
        i = 0
    else:
        i+= 1
#print(err)
print(w1_gd, w1)
print(np.sum((w1_gd - w1)**2))

[[ 1.4568168   2.05992186 -2.23902081 -0.17373297  0.53485372 -2.06938105
   0.31134896 -0.65413077 -0.67462562 -0.20905862]
 [ 1.16806365 -1.09095307  0.80564992 -0.32252143 -0.23929657 -0.1074663
   1.16602614  0.11077474 -0.03288695 -1.69082447]
 [ 0.91504145 -2.5872738  -1.96778559  1.30104101  0.45038408 -0.2982512
  -1.01236984 -0.93941806  0.59673385  0.53693557]
 [-0.16254485  0.26434734  0.69422489  0.8293075  -1.73101808 -2.84483155
  -0.00697974 -0.004139    1.49080002 -0.35665111]
 [-0.99911147 -0.99288859  2.47745427  0.08039793  0.32370474  1.36104625
  -0.45595408  1.62355215  2.50286373 -2.72827071]] [[ 1.20240403  0.99210315  1.31888047 -1.36127245  0.41883131 -0.13867465
   3.62902571 -0.07677168 -0.46350945  0.16674822]
 [-0.71108128 -0.37426968 -0.74617297  0.92257524 -1.15141099  0.76679516
  -0.1383068  -0.64993522 -0.17303843  0.32995932]
 [-1.32954492  1.32082858 -0.61375519  0.37244446  0.54970488  0.40155323
  -0.03549858  0.77839087  0.03161917 -0.84198578]
 